# 02 — Model Training & Evaluation
Train all three model types and compare performance.

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.utils.data_loader import load_sample_data, save_processed
from src.utils.preprocessor import Preprocessor
from src.models.explainable_model import ExplainableMedicalAI
from src.models.model_registry import ModelRegistry
from src.visualization.plots import plot_roc_curve, plot_confusion_matrix, plot_decision_tree

%matplotlib inline

In [ ]:
(X_train, X_test, y_train, y_test), feature_names = load_sample_data()

# Preprocess
prep = Preprocessor(feature_names)
X_train_s = prep.fit_transform(X_train)
X_test_s  = prep.transform(X_test)

# Save processed splits for reproducibility
save_processed(X_train_s, X_test_s, y_train, y_test, feature_names)
print(f'Train: {len(y_train)} | Test: {len(y_test)}')

In [ ]:
# Train all model types and compare
results = {}
for model_type in ['decision_tree', 'random_forest', 'logistic']:
    model = ExplainableMedicalAI(model_type=model_type)
    model.train(X_train_s, y_train, feature_names)
    metrics = model.evaluate(X_test_s, y_test)
    results[model_type] = metrics
    print(f'{model_type:20s}  Acc={metrics["accuracy"]:.4f}  AUC={metrics["roc_auc"]:.4f}')

In [ ]:
# Train best model (decision tree) and save
best = ExplainableMedicalAI(model_type='decision_tree')
best.train(X_train_s, y_train, feature_names)
best.save('../models/trained/decision_tree_v1.pkl')
prep.save('../models/trained/preprocessor.pkl')

# Register in registry
registry = ModelRegistry('../models/registry.json')
registry.register(
    name='cardiovascular_risk',
    path='models/trained/decision_tree_v1.pkl',
    metrics=best.evaluate(X_test_s, y_test),
    model_type='decision_tree'
)
print('Model saved and registered.')

In [ ]:
# ROC Curve
y_proba = best.predict_proba(X_test_s)
plot_roc_curve(y_test, y_proba, '../logs/roc_curve.png')

# Confusion Matrix
y_pred = best.predict(X_test_s)
plot_confusion_matrix(y_test, y_pred, '../logs/confusion_matrix.png')

# Decision Tree visualisation
plot_decision_tree(best.model, feature_names, output_path='../logs/decision_tree.png')
print('Evaluation plots saved to logs/')

In [ ]:
# Feature importance
imp = pd.Series(best.feature_importance()).sort_values(ascending=True)
imp.plot.barh(figsize=(8, 5), color='#63B3ED')
plt.title('Feature Importance (Decision Tree)')
plt.xlabel('Importance')
plt.tight_layout()
plt.savefig('../logs/feature_importance.png', dpi=120)
plt.show()